# 分板块研究平均偏度、峰度影响：主板

In [1]:
## 导入NumPy、Pandas、Wind
import numpy as np
import pandas as pd
from scipy import stats
from WindPy import w
import datetime
from dateutil.relativedelta import relativedelta

In [2]:
## 读入个股收益率各月的方差、偏度、峰度
prdct0 = pd.read_csv('Raw Transaction Data/prdct_dt.csv')
## 读入上市公司代码与上市板块
board = pd.read_csv('Raw Transaction Data/AllCorps.csv')
## 合并指标值与板块信息
prdct_mk = pd.merge(prdct0, board, left_on = 'code', right_on = 'codes', how = 'left')

In [3]:
## 启动Wind终端的API
## 判断WindPy是否已经登录成功
w.start() 
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2020 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [4]:
## 读取shibor数据
shibor = w.wsd("SHIBORON.IR", "open", "2007-01-01", "2019-01-31")
shibor = pd.DataFrame(shibor.Data,index = shibor.Codes,columns = shibor.Times)
shibor = shibor.T
shibor = shibor.reset_index()
shibor.columns = ['date', 'shibor']
shibor['shibor'] = shibor['shibor'] * 0.01
shibor['date'] = pd.to_datetime(shibor['date'])
shibor['year'] = shibor['date'].dt.year
shibor['month'] = shibor['date'].dt.month

In [5]:
## 计算各月shibor均值
intrst = shibor.groupby([shibor['year'], shibor['month']])['shibor'].agg([('mn_SHIBOR', 'mean'), ])
intrst = intrst.reset_index()
intrst = intrst.reset_index()
intrst.columns = ['index', 'year', 'month', 'shibor']

In [6]:
## 保留主板指标
prdct = prdct_mk[prdct_mk['boardtype'] == '主板']
prdct = prdct.reset_index(drop = True)
prdct = prdct.sort_values(by= ["code", 'year', 'month'] , ascending = True)

In [7]:
codes = prdct['code'].tolist()
codes = list(set(codes))
len(codes)

1904

## 计算领先一期的超额收益率

In [8]:
## 指数月收益数据
mkt_r = w.wsd("000002.SH", "pct_chg", "2007-01-01", "2019-01-31", "Period=M")
mkt_r = pd.DataFrame(mkt_r.Data, index = mkt_r.Codes, columns = mkt_r.Times)
mkt_r = mkt_r.T
mkt_r = mkt_r.reset_index()
mkt_r.columns = ['date', 'pct_chg']
mkt_r['date'] = pd.to_datetime(mkt_r['date'])
mkt_r['pct_chg'] = mkt_r['pct_chg'] * 0.01
mkt_r = mkt_r.reset_index()

In [9]:
## 计算月超额收益率
mkt_exr = pd.merge(mkt_r, intrst, on = 'index', how = 'left')
mkt_exr['exr'] = mkt_exr['pct_chg'] - mkt_exr['shibor']
mkt_exr = mkt_exr[['index', 'exr', 'year', 'month', 'date']]

In [10]:
mkt_r0 = mkt_exr.head(144)
mkt_r0.columns = ['index', 'r0', 'year', 'month', 'date']

In [11]:
## 删除首行，得到领先一期的收益率
mkt_exr = mkt_exr[['exr', 'year', 'month']]
mkt_exr = mkt_exr.drop(0, axis = 0)
mkt_exr = mkt_exr.reset_index(drop = True)
mkt_exr = mkt_exr.reset_index()

## 计算市场方差、偏度、峰度

In [12]:
## 获得上证A指的指数数据
trdt_mk = w.wsd("000002.SH", "pct_chg", "2007-01-01", "2019-01-31", "")
trdt_mk = pd.DataFrame(trdt_mk.Data, index = trdt_mk.Codes, columns = trdt_mk.Times)
trdt_mk = trdt_mk.T
trdt_mk = trdt_mk.reset_index()
trdt_mk.columns = ['date', 'pct_chg']
trdt_mk['pct_chg'] = trdt_mk['pct_chg'] * 0.01
trdt_mk['date'] = pd.to_datetime(trdt_mk['date'])
trdt_mk['year'] = trdt_mk['date'].dt.year
trdt_mk['month'] = trdt_mk['date'].dt.month

In [13]:
## 合并上证A指数据和shibor收益数据
mkt_dt = pd.merge(trdt_mk, shibor[['date', 'shibor']], on = 'date', how = 'left')
mkt_dt['exr'] = mkt_dt['pct_chg'] - mkt_dt['shibor']
mkt_dt = mkt_dt[['year', 'month', 'date', 'pct_chg', 'shibor', 'exr']]

In [14]:
## 分组，计算各月均值、方差、偏度、峰度
mkt_sta = mkt_dt.groupby([mkt_dt['year'], mkt_dt['month']])['exr'].agg([('num', 'count'), ('mean', 'mean'), ('math_var', 'var'), ('math_skew', 'skew'),('math_kurt', stats.kurtosis),])
mkt_sta = mkt_sta.reset_index()

In [15]:
## 删除首行
mkt_lf = mkt_dt.groupby([mkt_dt['year'], mkt_dt['month']])['date'].first()
mkt_lf = mkt_lf.reset_index()
mkt_lf.insert(2, 'flag', np.ones(len(mkt_lf)))
mkt_lf = mkt_lf[['date', 'flag']]

## 保留其他数据
mkt_wof = pd.merge(mkt_dt, mkt_lf, on = 'date', how = 'left')
mkt_wof = mkt_wof[mkt_wof['flag'] != 1]
mkt_wof = mkt_wof[['date', 'exr']]
mkt_wof = mkt_wof.rename(columns={'exr': 'exr_f'})
mkt_wof.insert(2, 'order', np.arange(len(mkt_wof)))

## 删除末行
mkt_ll = mkt_dt.groupby([mkt_dt['year'], mkt_dt['month']])['date'].last()
mkt_ll = mkt_ll.reset_index()
mkt_ll.insert(2, 'flag', np.ones(len(mkt_ll)))
mkt_ll = mkt_ll[['date', 'flag']]

## 保留其他数据
mkt_wol = pd.merge(mkt_dt, mkt_ll, on = 'date', how = 'left')
mkt_wol = mkt_wol[mkt_wol['flag'] != 1]
mkt_wol = mkt_wol[['year', 'month', 'exr']]
mkt_wol = mkt_wol.rename(columns={'exr': 'exr_l'})
mkt_wol.insert(2, 'order', np.arange(len(mkt_wol)))

## 合并去除第一行/最后一行的数据，准备计算调整的方差
## 注意，保留的日期系exr_f的日期
mkt_ajsta0 = pd.merge(mkt_wol, mkt_wof, on = 'order', how = 'left')
mkt_ajsta0 = mkt_ajsta0[[ 'year', 'month', 'date', 'exr_f', 'exr_l']]

## 合并均值数据
mkt_ajsta = pd.merge(mkt_ajsta0, mkt_sta[['year', 'month', 'mean']], on = ['year', 'month'], how = 'left')
mkt_ajsta['diff1'] = mkt_ajsta['exr_f'] - mkt_ajsta['mean']
mkt_ajsta['diff2'] = mkt_ajsta['exr_l'] - mkt_ajsta['mean']
mkt_ajsta['product'] = mkt_ajsta['diff1'] * mkt_ajsta['diff2']
mkt_ajsta = mkt_ajsta.groupby([mkt_ajsta['year'], mkt_ajsta['month']])['product'].agg([('prdct', 'sum'),])
mkt_ajsta = mkt_ajsta.reset_index()

## 合并调整后的数据和原统计指标值
mkt_ajs = pd.merge(mkt_sta, mkt_ajsta, on = ['year', 'month'], how = 'left' )
mkt_ajs['var'] = mkt_ajs['math_var'] * mkt_ajs['num'] + 2* mkt_ajs['prdct']
mkt_ajs['adj_var'] = mkt_ajs['var']/mkt_ajs['num']

## 保留数据
mkt_prdct = mkt_ajs[['year', 'month', 'num', 'mean', 'var', 'adj_var', 'math_skew', 'math_kurt']]
mkt_prdct = mkt_prdct[['year', 'month', 'var', 'math_skew', 'math_kurt']]
mkt_prdct.columns = ['year', 'month', 'mkt_var', 'mkt_skew', 'mkt_kurt']

In [16]:
mkt_prdct

,year,month,mkt_var,mkt_skew,mkt_kurt
0,2007,1,0.015597,-0.308346,-0.703897
1,2007,2,0.005015,-1.600126,2.212920
2,2007,3,0.001846,-0.748057,0.454222
3,2007,4,0.006406,-1.702514,3.315734
4,2007,5,0.006138,-2.031243,3.022892
...,...,...,...,...,...
140,2018,9,0.001017,0.431910,-0.691287
141,2018,10,0.005886,-0.348101,-0.169327
142,2018,11,0.001998,-0.151884,0.411607
143,2018,12,0.002404,0.604974,0.251528


## 等权重加权法

In [17]:
## 计算创业板平均方差、偏度与峰度
mnv_ew = prdct.groupby([prdct['year'], prdct['month']])['var', 'adj_var', 'math_skew', 'math_kurt'].mean()
mnv_ew = mnv_ew.reset_index()
mnv_ew = mnv_ew[['year', 'month', 'adj_var', 'math_skew', 'math_kurt']]
mnv_ew.columns = ['year', 'month', 'var_ew', 'skew_ew', 'kurt_ew']

In [18]:
mnv_ew

,year,month,var_ew,skew_ew,kurt_ew
0,2007,1,0.002133,-0.153232,-0.147689
1,2007,2,0.000953,-0.509944,0.671196
2,2007,3,0.001507,0.095211,0.153316
3,2007,4,0.006810,-0.246183,0.649273
4,2007,5,0.003254,-0.337993,0.110268
...,...,...,...,...,...
139,2018,8,0.000662,0.332867,0.496296
140,2018,9,0.000384,0.074051,0.346407
141,2018,10,0.001332,-0.458173,0.719020
142,2018,11,0.000832,-0.041235,0.657091


## 市值加权法

In [19]:
## 读取市值数据
evdt = w.wsd(codes, "ev", "2006-12-01", "2018-11-30", "unit=1;Period=M;TradingCalendar=SZSE")
ev = pd.DataFrame(evdt.Data, index = evdt.Codes, columns = evdt.Times)
ev = ev.T
ev = ev.reset_index()

In [20]:
## 转为一维表，准备合并
ev_T = pd.melt(ev, 
            id_vars = 'index', 
            value_vars = list(ev.columns[1:]), 
            var_name='code', 
            value_name='ev_month')
ev_T.columns = ['date', 'code', 'ev_month']
ev_T['date'] = pd.to_datetime(ev_T['date'])
ev_T['date'] = ev_T['date'].apply(lambda x: x + relativedelta(months = +1)) 
## 生成年、月
ev_T['year'] = ev_T['date'].dt.year
ev_T['month'] = ev_T['date'].dt.month

In [21]:
## 合并prdct和ev_w
vw_fore = pd.merge(prdct, ev_T, on = ['code', 'year', 'month'], how = 'left')
vw_fore = vw_fore.dropna().reset_index(drop = True)

## 按月分组，计算各月总市值
## 计算样本数据的权重
tt_ev = vw_fore.groupby([vw_fore['year'], vw_fore['month']])['ev_month'].agg([('TEV', 'sum'),])
tt_ev = tt_ev.reset_index()
ev_w = pd.merge(vw_fore, tt_ev, on = ['year', 'month'], how = 'left')
ev_w['weight'] = ev_w['ev_month']/ ev_w['TEV']
ev_w = ev_w.sort_values(by= ["code", 'year', 'month'] , ascending = True)

In [22]:
ev_w

,code,year,month,num,var,adj_var,math_skew,math_kurt,codes,boardtype,date,ev_month,TEV,weight
0,000001.SZ,2007,1,20,0.046588,0.002329,-0.897868,-0.690205,000001.SZ,主板,2007-01-29,2.815605e+10,9.576136e+12,0.002940
1,000001.SZ,2007,2,15,0.027840,0.001856,0.168187,-1.414870,000001.SZ,主板,2007-02-28,3.722358e+10,1.117186e+13,0.003332
2,000001.SZ,2007,3,22,0.009971,0.000453,0.366919,-0.376840,000001.SZ,主板,2007-03-28,3.706791e+10,1.177925e+13,0.003147
3,000001.SZ,2007,4,21,0.025306,0.001205,-0.734114,-0.084518,000001.SZ,主板,2007-04-30,3.673712e+10,1.333896e+13,0.002754
4,000001.SZ,2007,5,18,0.003345,0.000186,0.307989,-0.024779,000001.SZ,主板,2007-05-30,5.049408e+10,1.660706e+13,0.003041
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206475,603999.SH,2018,8,23,0.011140,0.000484,0.517311,-0.263385,603999.SH,主板,2018-08-31,3.398400e+09,4.163145e+13,0.000082
206476,603999.SH,2018,9,19,0.002344,0.000123,-0.664188,-0.413938,603999.SH,主板,2018-09-30,3.231360e+09,3.964114e+13,0.000082
206477,603999.SH,2018,10,18,0.073409,0.004078,0.159128,0.074828,603999.SH,主板,2018-10-28,2.943360e+09,4.089298e+13,0.000072
206478,603999.SH,2018,11,22,0.015331,0.000697,-0.131413,-0.705213,603999.SH,主板,2018-11-30,2.776320e+09,3.768550e+13,0.000074


In [23]:
## 筛选需要的指标列
vw = ev_w[['code', 'year', 'month', 'weight', 'var', 'adj_var', 'math_skew', 'math_kurt']]

## 计算加权指标
vw['var_vw'] = vw['adj_var'] * vw['weight']
vw['skew_vw'] = vw['math_skew'] * vw['weight']
vw['kurt_vw'] = vw['math_kurt'] * vw['weight']

## 按月加总
mnv_vw = vw.groupby([vw['year'], vw['month']])['var_vw', 'skew_vw', 'kurt_vw'].sum()
mnv_vw = mnv_vw.reset_index()
mnv_vw = mnv_vw[['year', 'month', 'var_vw', 'skew_vw', 'kurt_vw']]

## 合并数据

In [24]:
mkt_reg = pd.merge(mkt_exr[['index', 'exr']], mkt_r0, on = 'index', how = 'left')
mkt_reg = pd.merge(mkt_reg, mkt_prdct, on = ['year', 'month'], how = 'left')
mkt_reg = pd.merge(mkt_reg, mnv_ew, on = ['year', 'month'], how = 'left')
mkt_reg = pd.merge(mkt_reg, mnv_vw, on = ['year', 'month'], how = 'left')
mkt_reg = mkt_reg.drop(['index', 'year', 'month'], axis = 1)
mkt_reg = mkt_reg.dropna()
mkt_reg.columns = ['r', 'r0', 'date','mkt_var', 'mkt_skew', 'mkt_kurt', 'var_ew', 'skew_ew', 'kurt_ew', 'var_vw', 'skew_vw', 'kurt_vw']
mkt_reg = mkt_reg.reset_index(drop = True)

In [25]:
mkt_reg[mkt_reg['date'] < '2015-01-01'].to_csv("Plot/reg_mkt1.csv", index = False) 